# App Review Scraper

In [ ]:
from google_play_scraper import Sort, reviews_all
import pandas as pd
import os
import time

APP_ID = 'com.telkom.tracencare'

# Path adjusted for notebook being inside 'notebooks' folder
OUTPUT_DIR = '../data/raw'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'satu_sehat_reviews.csv')

# Create data/raw directory if not exists
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"[✓] Created directory: {OUTPUT_DIR}")

print(f"[i] App ID: {APP_ID}")
print(f"[i] Output file: {OUTPUT_FILE}")

[i] App ID: com.telkom.tracencare
[i] Output file: ../data/raw/satu_sehat_reviews.csv


In [2]:
print(f"Fetching reviews for: {APP_ID}...")

# 1. Fetching Indonesian reviews
print("[*] Collecting Indonesian (id) reviews...")
res_id = reviews_all(
    APP_ID,
    lang='id',
    country='id',
    sort=Sort.NEWEST
)

# 2. Fetching English reviews (to catch users with English phone settings)
print("[*] Collecting English (en) reviews...")
res_en = reviews_all(
    APP_ID,
    lang='en',
    country='id',
    sort=Sort.NEWEST
)

# Merge and remove duplicates based on reviewId
all_reviews = res_id + res_en
df = pd.DataFrame(all_reviews)
if not df.empty:
    df.drop_duplicates(subset=['reviewId'], inplace=True)
    print(f"[✓] Successfully collected {len(df)} unique text reviews.")
else:
    print("[!] No reviews found.")

Fetching reviews for: com.telkom.tracencare...
[*] Collecting Indonesian (id) reviews...
[*] Collecting English (en) reviews...
[✓] Successfully collected 448387 unique text reviews.


In [3]:
# Final processing and saving
if not df.empty:
    try:
        # Select relevant columns that exist in the data
        available_columns = df.columns.tolist()
        print(f"[i] Available columns: {available_columns}\n")
        
        # Map columns carefully to handle variations
        df_final = df[[
            'userName', 'score', 'at', 'content', 
            'thumbsUpCount', 'reviewCreatedVersion'
        ]].copy()
        
        df_final.columns = [
            'user_name', 'rating', 'date', 'review_text', 
            'likes', 'app_version'
        ]
        
        # Save to CSV
        df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"[✓] Results saved to: {OUTPUT_FILE}")
        print(f"[✓] Total reviews saved: {len(df_final)}")
        
        print("\n[i] Data summary:")
        print(f"    └─ Rating distribution:\n{df_final['rating'].value_counts().sort_index()}")
        print(f"\n    └─ First 5 reviews:")
        print(df_final.head())
        
    except KeyError as e:
        print(f"[!] Error: Missing column {e}")
        print(f"    Available columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"[!] Error saving file: {str(e)}")
else:
    print("[!] No data to process.")

[i] Available columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

[✓] Results saved to: ../data/raw/satu_sehat_reviews.csv
[✓] Total reviews saved: 448387

[i] Data summary:
    └─ Rating distribution:
rating
1    176486
2     30389
3     29360
4     29579
5    182573
Name: count, dtype: int64

    └─ First 5 reviews:
         user_name  rating                date       review_text  likes  \
0  Pengguna Google       5 2026-05-04 18:56:35       sudah bagus      0   
1  Pengguna Google       1 2026-05-04 17:26:39        apk sampah      0   
2  Pengguna Google       5 2026-05-04 15:10:05        bagussssss      0   
3  Pengguna Google       5 2026-05-04 09:20:26   Sangat membantu      0   
4  Pengguna Google       4 2026-05-04 08:41:01  susah digunakan.      0   

  app_version  
0       8.1.0  
1       8.7.1  
2         NaN  
3       8.7.1  
4       8.7.1  
